###**Data reading**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.format('parquet').load('abfss://bronze@databrcks.dfs.core.windows.net/customers')
display(df.limit(10))

customer_id,first_name,last_name,email,city,state,_rescued_data
C00001,Emily,Mooney,rushjeff@ryan.org,Johnsonmouth,MS,null
C00002,Andrea,Sellers,mccoykiara@kelly.com,Stephenfort,WY,null
C00003,Craig,Hayes,rebeccamiller@yahoo.com,South Stephenshire,LA,null
C00004,Bryan,Scott,lawrence05@campbell.info,Chrisland,ND,null
C00005,Sean,Vasquez,carrie45@yahoo.com,East Dennistown,RI,null
C00006,Kevin,Mccarthy,traceyramos@gmail.com,North Matthew,IN,null
C00007,Amanda,Doyle,scottallen@gmail.com,Joneshaven,VA,null
C00008,Paul,Campos,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,null
C00009,Mary,Green,dennis03@yahoo.com,Kimberlyview,MD,null
C00010,James,Myers,charles58@murillo.net,West Hector,OK,null


In [0]:
df = df.drop('_rescued_data')

customer_id,first_name,last_name,email,city,state,domains,full_name
C00001,Emily,Mooney,rushjeff@ryan.org,Johnsonmouth,MS,ryan.org,Emily Mooney
C00002,Andrea,Sellers,mccoykiara@kelly.com,Stephenfort,WY,kelly.com,Andrea Sellers
C00003,Craig,Hayes,rebeccamiller@yahoo.com,South Stephenshire,LA,yahoo.com,Craig Hayes
C00004,Bryan,Scott,lawrence05@campbell.info,Chrisland,ND,campbell.info,Bryan Scott
C00005,Sean,Vasquez,carrie45@yahoo.com,East Dennistown,RI,yahoo.com,Sean Vasquez
C00006,Kevin,Mccarthy,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy
C00007,Amanda,Doyle,scottallen@gmail.com,Joneshaven,VA,gmail.com,Amanda Doyle
C00008,Paul,Campos,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,horton-adams.com,Paul Campos
C00009,Mary,Green,dennis03@yahoo.com,Kimberlyview,MD,yahoo.com,Mary Green
C00010,James,Myers,charles58@murillo.net,West Hector,OK,murillo.net,James Myers


In [0]:
df = df.withColumn("domains", split(col("email"),'@')[1])

customer_id,first_name,last_name,email,city,state,domains
C00001,Emily,Mooney,rushjeff@ryan.org,Johnsonmouth,MS,ryan.org
C00002,Andrea,Sellers,mccoykiara@kelly.com,Stephenfort,WY,kelly.com
C00003,Craig,Hayes,rebeccamiller@yahoo.com,South Stephenshire,LA,yahoo.com
C00004,Bryan,Scott,lawrence05@campbell.info,Chrisland,ND,campbell.info
C00005,Sean,Vasquez,carrie45@yahoo.com,East Dennistown,RI,yahoo.com
C00006,Kevin,Mccarthy,traceyramos@gmail.com,North Matthew,IN,gmail.com
C00007,Amanda,Doyle,scottallen@gmail.com,Joneshaven,VA,gmail.com
C00008,Paul,Campos,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,horton-adams.com
C00009,Mary,Green,dennis03@yahoo.com,Kimberlyview,MD,yahoo.com
C00010,James,Myers,charles58@murillo.net,West Hector,OK,murillo.net


In [0]:
df.groupBy("domains").agg(count('customer_id').alias('top_domains')).sort('top_domains', ascending=False).display()

domains,top_domains
gmail.com,374
hotmail.com,360
yahoo.com,331
brown.com,8
davis.com,8
smith.com,7
johnson.com,5
hernandez.com,5
kennedy.com,4
white.com,4


In [0]:
gmail_users = df.filter(col('domains')=='gmail.com')

customer_id,first_name,last_name,email,city,state,domains
C00006,Kevin,Mccarthy,traceyramos@gmail.com,North Matthew,IN,gmail.com
C00007,Amanda,Doyle,scottallen@gmail.com,Joneshaven,VA,gmail.com
C00011,Jacob,Le,anita65@gmail.com,Houstonfurt,AR,gmail.com
C00012,Chad,Banks,beardtravis@gmail.com,East Jenniferview,WI,gmail.com
C00013,James,Martin,jwood@gmail.com,Lake Gregoryshire,OR,gmail.com
C00017,Angela,Carter,rhondaferguson@gmail.com,West Loriborough,GA,gmail.com
C00028,Barry,Baker,steven27@gmail.com,North Gregoryfurt,NM,gmail.com
C00030,Holly,Collins,cfuller@gmail.com,Port Coltonton,AL,gmail.com
C00032,Alexandria,Singleton,egallegos@gmail.com,Samuelshire,KS,gmail.com
C00034,Carol,Matthews,stacy78@gmail.com,Gregorybury,IA,gmail.com


In [0]:
df = df.withColumn('full_name', concat(col('first_name'),lit(' '),col('last_name')))
df = df.drop('first_name','last_name')
display(df.limit(10))

customer_id,email,city,state,domains,full_name
C00001,rushjeff@ryan.org,Johnsonmouth,MS,ryan.org,Emily Mooney
C00002,mccoykiara@kelly.com,Stephenfort,WY,kelly.com,Andrea Sellers
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,yahoo.com,Craig Hayes
C00004,lawrence05@campbell.info,Chrisland,ND,campbell.info,Bryan Scott
C00005,carrie45@yahoo.com,East Dennistown,RI,yahoo.com,Sean Vasquez
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy
C00007,scottallen@gmail.com,Joneshaven,VA,gmail.com,Amanda Doyle
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,horton-adams.com,Paul Campos
C00009,dennis03@yahoo.com,Kimberlyview,MD,yahoo.com,Mary Green
C00010,charles58@murillo.net,West Hector,OK,murillo.net,James Myers


###**Data writing**

In [0]:
df.write.mode('append').format('delta').save('abfss://silver@databrcks.dfs.core.windows.net/customers')